In [1]:
from atomica.models import MultiClassClassifierModel, MultiLabelClassifierModel, ResidueClassifierModel
from atomica.data.dataset import MultiClassLabelledPDBDataset, LabelledPDBDataset
from atomica.trainers import Trainer

from multiclass_metrics import compute_multiclass_metrics
from multilabel_metrics import compute_multilabel_metrics

from torch.utils.data import DataLoader
import torch
from tqdm import tqdm
import numpy as np
import pandas as pd
import os
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score

DATA_DIR="/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks"
MODEL_DIR="/n/netscratch/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_benchmark"

/n/home13/afang/.conda/envs/interactenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# RNAGo (Not SOTA, may need joint model)

Validation loss is a better metric than AUPRC for this task.

Not SOTA compared to RNAFM

RNAFM best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/RNAFM/RNAGo/version_2/test_metrics.json`
* subset_accuracy: 0.8267
* f1_macro: 0.7384
* f1_micro: 0.7872
* f1_weighted: 0.7980

RNAErnie best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rnaernie/RNAGo/version_1/test_metrics.json`
* subset_accuracy 0.7466
* f1_macro 0.6567
* f1_micro 0.7097
* f1_weighted 0.7128

RiNAlmo best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rinalmo/RNAGo/version_5/test_metrics.json`
* subset_accuracy: 0.92
* f1_macro 0.8712,
* f1_micro 0.9149,
* f1_weighted 0.9202,


In [194]:
# model_checkpoint = f"{MODEL_DIR}/RNAGo/models/version_22/checkpoint/epoch118_step18207.pt"
# model_checkpoint = f"{MODEL_DIR}/RNAGo/models/version_17/checkpoint/epoch294_step45135.pt"
# model_checkpoint = f"{MODEL_DIR}/RNAGo/models/version_28/checkpoint/epoch362_step55539.pt" # SOTA 

# grad clip 1.0, lr 2e-5
# model_checkpoint1 = f"{MODEL_DIR}/RNAGo/models/version_32/checkpoint/epoch173_step26622.pt" # SOTA 2, lowest val loss
# model_checkpoint = f"{MODEL_DIR}/RNAGo/models/version_49/checkpoint/epoch217_step32918.pt"

# grad clip 0.5, lr 2e-5
model_checkpoint = f"{MODEL_DIR}/RNAGo/models/version_42/checkpoint/epoch164_step25245.pt" # SOTA 3, lowest val loss
model_checkpoint = f"{MODEL_DIR}/RNAGo/models/version_45/checkpoint/epoch181_step27118.pt" 
model_checkpoint = f"{MODEL_DIR}/RNAGo/models/version_48/checkpoint/epoch219_step32780.pt"


def get_model(model_checkpoint: str) -> str:
    model_config = os.path.join(os.path.dirname(model_checkpoint), "config.json")
    model = MultiLabelClassifierModel.load_from_config_and_weights(model_config, model_checkpoint)
    return model

def get_atomica_results(model_checkpoint, split="val"):
    atomica_preds = []
    dataset = MultiClassLabelledPDBDataset(f"{DATA_DIR}/RNAGo/RNAGo_{split}_processed.parquet")
    model = get_model(model_checkpoint)

    model.eval()
    model.to("cuda")
    batch_size = 1
    for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset) // batch_size):
        with torch.no_grad():
            batch = [dataset[j] for j in range(i, min(i+batch_size, len(dataset)))]
            batch = MultiClassLabelledPDBDataset.collate_fn(batch)
            batch = Trainer.to_device(batch, "cuda")
            atomica_preds.append(model.infer(batch).cpu().numpy())
    atomica_preds = np.concatenate(atomica_preds)
    atomica_labels = np.array([x['label'] for x in dataset.data])
    atomica_results = pd.DataFrame({
        'id': [x['id'] for x in dataset.data],
        'label': [atomica_labels[i] for i in range(len(atomica_labels))],
        'pred_probability': [atomica_preds[i] for i in range(len(atomica_preds))],
    })
    atomica_results['pred'] = atomica_results['pred_probability'].apply(lambda x: (x > 0.5).astype(int))
    return atomica_results

atomica_results_val = get_atomica_results(model_checkpoint, split="val")
atomica_results_test = get_atomica_results(model_checkpoint, split="test")


/n/home13/afang/.conda/envs/interactenv/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
100%|██████████| 75/75 [00:17<00:00,  4.39it/s]
/n/home13/afang/.conda/envs/interactenv/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
100%|██████████| 75/75 [00:21<00:00,  3.54it/s]


In [195]:
best_f1_threshold = 0.0
best_f1_score = 0.0
f1_thresholds = np.linspace(0.0, 0.5, 6)
for threshold in f1_thresholds:
    metrics = compute_multilabel_metrics(
        y_true=np.stack(atomica_results_val['label']),
        y_proba=np.stack(atomica_results_val['pred_probability']),
        threshold=threshold
    )
    if metrics.f1_weighted > best_f1_score:
        best_f1_score = metrics.f1_weighted
        best_f1_threshold = threshold
print(best_f1_threshold)

0.1


In [196]:
metrics = compute_multilabel_metrics(
    y_true=np.stack(atomica_results_test['label']),
    y_proba=np.stack(atomica_results_test['pred_probability']),
    threshold=best_f1_threshold
)

In [197]:
metrics

MultilabelMetricsResult(subset_accuracy=0.5733333333333334, f1_macro=0.6707471931862174, f1_micro=0.6391752577319587, f1_weighted=0.6539707251175855, f1_samples=0.31911111111111107, jaccard_macro=0.5504329004329005, jaccard_micro=0.4696969696969697, jaccard_weighted=0.5359445519019986, jaccard_samples=0.3111111111111111, roc_auc_ovr_macro=0.915263485855672, roc_auc_ovr_weighted=0.8283042702632022, roc_auc_ovr_micro=0.9373378308251168, per_label={0: {'precision': 0.7, 'recall': 0.875, 'f1': 0.7777777777777778, 'support': 8.0, 'jaccard': 0.6363636363636364}, 1: {'precision': 1.0, 'recall': 0.7142857142857143, 'f1': 0.8333333333333334, 'support': 7.0, 'jaccard': 0.7142857142857143}, 2: {'precision': 1.0, 'recall': 0.9090909090909091, 'f1': 0.9523809523809523, 'support': 11.0, 'jaccard': 0.9090909090909091}, 3: {'precision': 0.25, 'recall': 1.0, 'f1': 0.4, 'support': 1.0, 'jaccard': 0.25}, 4: {'precision': 0.38095238095238093, 'recall': 0.4, 'f1': 0.3902439024390244, 'support': 20.0, 'jacc

In [198]:
metrics.f1_weighted

0.6539707251175855

# RNA Ligand (Not SOTA)
May need joint model for this task.

The balanced sampling helps. Not SOTA.
TODO: diversify the dataset by randomly adding / removing residues from the pocket.

RNAFM best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/RNAFM/RNA_Ligand/version_1/test_metrics.json`
* accuracy: 0.9545
* balanced_accuracy: 0.9333
* f1_macro: 0.9270
* f1_micro: 0.9545
* f1_weighted: 0.9532

RNAErnie best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rnaernie/RNA_Ligand/version_1/test_metrics.json`
* accuracy 0.9318
* balanced_accuracy 0.9103
* f1_macro 0.9151
* f1_micro 0.9318
* f1_weighted 0.9320

RiNAlmo best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rinalmo/RNA_Ligand/version_18/test_metrics.json`
* accuracy: 0.7045,
* balanced_accuracy: 0.4874,
* f1_macro: 0.4646
* f1_micro: 0.7045
* f1_weighted: 0.6595,

In [ ]:
# model_checkpoint = f"{MODEL_DIR}/RNA_Ligand/models/version_11/checkpoint/epoch149_step6600.pt" # 0.75 acc, lr 3e-5
# model_checkpoint = f"{MODEL_DIR}/RNA_Ligand/models/version_22/checkpoint/epoch1303_step59984.pt"
# model_checkpoint = f"{MODEL_DIR}/RNA_Ligand/models/version_27/checkpoint/epoch171_step7568.pt" # 0.86 acc 
model_checkpoint = f"{MODEL_DIR}/RNA_Ligand/models/version_34/checkpoint/epoch205_step9064.pt" # lowest val loss, 0.86 acc 
model_checkpoint = f"{MODEL_DIR}/RNA_Ligand/models/version_40/checkpoint/epoch217_step9592.pt" # AUPRC, 0.86 acc 

def get_model(model_checkpoint: str) -> str:
    model_config = os.path.join(os.path.dirname(model_checkpoint), "config.json")
    model = MultiClassClassifierModel.load_from_config_and_weights(model_config, model_checkpoint)
    return model

dataset = MultiClassLabelledPDBDataset(f"{DATA_DIR}/RNA_Ligand/RNA_Ligand_test_processed.parquet")
model = get_model(model_checkpoint)

atomica_preds = []
model.eval()
model.to("cuda")
batch_size = 1
for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset) // batch_size):
    with torch.no_grad():
        batch = [dataset[j] for j in range(i, min(i+batch_size, len(dataset)))]
        batch = MultiClassLabelledPDBDataset.collate_fn(batch)
        batch = Trainer.to_device(batch, "cuda")
        atomica_preds.append(model.infer(batch).cpu().numpy())
atomica_preds = np.concatenate(atomica_preds)
atomica_labels = np.array([x['label'] for x in dataset.data])
pred_indxes = np.argmax(atomica_preds, axis=1)
atomica_results = pd.DataFrame({
    'id': [x['id'] for x in dataset.data],
    'label': atomica_labels,
    'pred': pred_indxes,
    'pred_probability': [atomica_preds[i] for i in range(len(atomica_preds))],
})

/n/home13/afang/.conda/envs/interactenv/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
100%|██████████| 44/44 [00:10<00:00,  4.28it/s]


In [37]:
rnafm_probabilities = np.load("/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/RNAFM/RNA_Ligand/version_1/test_probabilities.npy")
atomica_results['rnafm_probability'] = list(rnafm_probabilities)
atomica_results['ensemble_probability'] = 0.5 * atomica_results['rnafm_probability'] + 0.5 * atomica_results['pred_probability']
atomica_results['ensemble_pred'] = atomica_results['ensemble_probability'].apply(lambda x: np.argmax(x))

In [17]:
atomica_results['label'].value_counts()

label
1    29
0    10
2     5
Name: count, dtype: int64

In [18]:
atomica_results['pred'].value_counts()

pred
1    32
0    12
Name: count, dtype: int64

In [19]:
np.mean(atomica_results['label'] == atomica_results['pred'])

0.8181818181818182

In [38]:
metrics = compute_multiclass_metrics(
    np.array(atomica_results['label']),
    np.array(atomica_results['pred']),
    np.stack(atomica_results['pred_probability']),
    [0,1,2],
)
metrics

MetricsResult(accuracy=0.7954545454545454, balanced_accuracy=0.5988505747126437, f1_macro=0.5348627980206927, f1_micro=0.7954545454545454, f1_weighted=0.758618574408048, jaccard_macro=0.4560404807084124, jaccard_micro=0.660377358490566, jaccard_weighted=0.6731067793686389, roc_auc_ovr_macro=0.9084291187739465, roc_auc_ovr_weighted=0.9348484848484848, roc_auc_ovo_macro=0.8224137931034483, roc_auc_ovo_weighted=0.889557210031348, per_class={0: {'precision': 0.5625, 'recall': 0.9, 'f1': 0.6923076923076923, 'support': 10.0, 'jaccard': 0.5294117647058824}, 1: {'precision': 0.9285714285714286, 'recall': 0.896551724137931, 'f1': 0.9122807017543859, 'support': 29.0, 'jaccard': 0.8387096774193549}, 2: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 5.0, 'jaccard': 0.0}}, per_class_ovr_auc={0: 0.9, 1: 0.9586206896551724, 2: 0.8666666666666667})

In [40]:
metrics = compute_multiclass_metrics(
    np.array(atomica_results['label']),
    np.array(atomica_results['ensemble_pred']),
    np.stack(atomica_results['ensemble_probability']),
    [0,1,2],
)
metrics

MetricsResult(accuracy=0.8181818181818182, balanced_accuracy=0.6103448275862069, f1_macro=0.5557894736842105, f1_micro=0.8181818181818182, f1_weighted=0.7880382775119618, jaccard_macro=0.4875, jaccard_micro=0.6923076923076923, jaccard_weighted=0.7210227272727273, roc_auc_ovr_macro=0.9791361106776929, roc_auc_ovr_weighted=0.9867030028794734, roc_auc_ovo_macro=0.9554022988505748, roc_auc_ovo_weighted=0.9738244514106582, per_class={0: {'precision': 0.6, 'recall': 0.9, 'f1': 0.72, 'support': 10.0, 'jaccard': 0.5625}, 1: {'precision': 0.9642857142857143, 'recall': 0.9310344827586207, 'f1': 0.9473684210526315, 'support': 29.0, 'jaccard': 0.9}, 2: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 5.0, 'jaccard': 0.0}}, per_class_ovr_auc={0: 0.9676470588235295, 1: 0.9954022988505747, 2: 0.9743589743589743})

# RNA-Protein (SOTA)
This also beats RNAGlib 

RNAFM best metrics: 
`/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/RNAFM/RNA_Protein/version_7/test_metrics.json`
* accuracy 0.6827
* balanced_accuracy 0.5438
* roc_auc 0.6108
* auprc 0.3631

RNAErnie best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rnaernie/RNA_Protein/version_6/test_metrics.json`
* accuracy: 0.6853,
* balanced_accuracy": 0.5701,
* roc_auc: 0.6317
* auprc: 0.3743

RNAProtein best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rinalmo/RNA_Protein/version_23/test_metrics.json`
* accuracy: 0.6749,
* balanced_accuracy: 0.5711,
* roc_auc: 0.6277,
* auprc: 0.3748,

In [ ]:
model_checkpoint = f"{MODEL_DIR}/RNA_Protein/models/version_1/checkpoint/epoch14_step3210.pt"

def get_model(model_checkpoint: str) -> str:
    model_config = os.path.join(os.path.dirname(model_checkpoint), "config.json")
    model = ResidueClassifierModel.load_from_config_and_weights(model_config, model_checkpoint)
    return model

dataset = LabelledPDBDataset(f"{DATA_DIR}/RNA_Protein/RNA_Protein_test_processed.parquet")
model = get_model(model_checkpoint)

atomica_preds = []
model.eval()
model.to("cuda")
batch_size = 1
for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset) // batch_size):
    with torch.no_grad():
        batch = [dataset[j] for j in range(i, min(i+batch_size, len(dataset)))]
        batch = LabelledPDBDataset.collate_fn(batch)
        batch = Trainer.to_device(batch, "cuda")
        atomica_preds.append(model.infer(batch).cpu().numpy())
atomica_preds = np.concatenate(atomica_preds).flatten()
atomica_labels = np.concatenate([x['label'] for x in dataset.data])
atomica_ids = sum([[x['id']] * len(x['label']) for x in dataset.data], [])

atomica_results = pd.DataFrame({
    'id': atomica_ids,
    'label': atomica_labels,
    'pred': atomica_preds,
})
atomica_results

In [34]:
auroc = roc_auc_score(atomica_labels, atomica_preds)

precision, recall, thresholds = precision_recall_curve(atomica_labels, atomica_preds)
auprc = auc(recall, precision)

print("AUROC: ", auroc)
print("AUPRC: ", auprc)
print("Mean label: ", np.mean(atomica_labels))

AUROC:  0.747219629920314
AUPRC:  0.5440883585916663
Mean label:  0.274076122016404


In [70]:
thresholds = np.linspace(0.0, 1.0, 101)  # e.g. test thresholds from 0.00 to 1.00
f1s = [f1_score(atomica_labels, (atomica_preds >= t).astype(int)) for t in thresholds]

best_t = thresholds[np.argmax(f1s)]
best_f1 = max(f1s)

print(f"Best threshold = {best_t:.3f}, Best F1 = {best_f1:.4f}")

Best threshold = 0.220, Best F1 = 0.5430


### Grouped scores by id

In [71]:
atomica_results['pred_bin'] = atomica_preds >= best_t

atomica_results['correct'] = atomica_results['label'] == atomica_results['pred_bin']
balanced_acc = atomica_results.groupby('id')['correct'].mean().mean()

atomica_results['baseline_correct'] = atomica_results['label'] == 0
baseline_acc = atomica_results.groupby('id')['baseline_correct'].mean().mean()

print(balanced_acc, baseline_acc)

0.6928521605484866 0.6970169462442661


In [72]:
grouped = atomica_results.groupby('id')

auprcs, aurocs = [], []
baseline = []

for _, g in grouped:
    y_true = g['label']
    y_pred = g['pred']
    # skip if only one class is present (AUROC undefined)
    if len(set(y_true)) > 1:
        precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
        auprc = auc(recall, precision)
        auroc = roc_auc_score(y_true, y_pred)
        auprcs.append(auprc)
        aurocs.append(auroc)
        baseline.append(np.mean(y_true))


mean_auprc = sum(auprcs) / len(auprcs)
mean_auroc = sum(aurocs) / len(aurocs)
mean_baseline = sum(baseline) / len(baseline)

print(f"Mean AUPRC: {mean_auprc:.4f}")
print(f"Mean AUROC: {mean_auroc:.4f}")
print(f"Baseline: {mean_baseline:.4f}")

Mean AUPRC: 0.5396
Mean AUROC: 0.5999
Baseline: 0.4230


# RNA-Site (not SOTA)

RNA FM best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/RNAFM/RNA_Site/version_5/test_metrics.json`
* accuracy 0.9209
* balanced_accuracy 0.4995
* roc_auc 0.6639
* auprc 0.1375

RNAErnie best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rnaernie/RNA_Site/version_0/test_metrics.json`
* accuracy: 0.9228
* balanced_accuracy: 0.5059
* roc_auc: 0.5980
* auprc: 0.1816

RiNAlmo best metrics: `/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rinalmo/RNA_Site/version_22/test_metrics.json`
* accuracy: 0.9242,
* balanced_accuracy: 0.5283,
* roc_auc: 0.6887,
* auprc: 0.2197,




In [ ]:
model_checkpoint = f"{MODEL_DIR}/RNA_Site/models/version_10/checkpoint/epoch51_step2444.pt" # SOTA, lowest val loss

def get_model(model_checkpoint: str) -> str:
    model_config = os.path.join(os.path.dirname(model_checkpoint), "config.json")
    model = ResidueClassifierModel.load_from_config_and_weights(model_config, model_checkpoint)
    return model

dataset = LabelledPDBDataset(f"{DATA_DIR}/RNA_Site/RNA_Site_test_processed.parquet")
model = get_model(model_checkpoint)

atomica_preds = []
model.eval()
model.to("cuda")
batch_size = 1
for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset) // batch_size):
    with torch.no_grad():
        batch = [dataset[j] for j in range(i, min(i+batch_size, len(dataset)))]
        batch = LabelledPDBDataset.collate_fn(batch)
        batch = Trainer.to_device(batch, "cuda")
        atomica_preds.append(model.infer(batch).cpu().numpy())
atomica_preds = np.concatenate(atomica_preds).flatten()
atomica_labels = np.concatenate([x['label'] for x in dataset.data])
atomica_ids = []
for x in dataset.data:
    assert len(x['label']) == len(x['block_to_pdb_indexes'])
    for _, pdb_index in sorted(x['block_to_pdb_indexes'].items()):
        atomica_ids.append(x['id'] + '_' + str(pdb_index))

atomica_results = pd.DataFrame({
    'id': atomica_ids,
    'label': atomica_labels,
    'pred': atomica_preds,
})

/n/home13/afang/.conda/envs/interactenv/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
100%|██████████| 33/33 [00:09<00:00,  3.33it/s]


In [58]:
auroc = roc_auc_score(atomica_labels, atomica_preds)

precision, recall, thresholds = precision_recall_curve(atomica_labels, atomica_preds)
auprc = auc(recall, precision)

f1 = f1_score(atomica_labels, atomica_preds >= 0.3)

print("AUROC: ", auroc)
print("AUPRC: ", auprc)
print("F1: ", f1)
print("Mean label: ", np.mean(atomica_labels))

AUROC:  0.6472588185295367
AUPRC:  0.20539200432441929
F1:  0.23346303501945526
Mean label:  0.07824074074074074


In [59]:
thresholds = np.linspace(0.0, 1.0, 101)  # e.g. test thresholds from 0.00 to 1.00
f1s = [f1_score(atomica_labels, (atomica_preds >= t).astype(int)) for t in thresholds]

best_t = thresholds[np.argmax(f1s)]
best_f1 = max(f1s)

print(f"Best threshold = {best_t:.3f}, Best F1 = {best_f1:.4f}")

Best threshold = 0.150, Best F1 = 0.2680


In [111]:
sequences_dataset = pd.read_parquet(f"{DATA_DIR}/RNA_Site/RNA_Site_test_input.parquet")
rnafm_preds = np.load("/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/rnaglib_tasks/models/rinalmo/RNA_Site/version_22/test_probabilities.npy")[:, 1]

sequences_ids = []
for x in sequences_dataset.itertuples():
    for i in x.pdb_indexes:
        sequences_ids.append(x.pdb_id + '_' + i)
len(sequences_ids)

sequences_preds = pd.DataFrame({
    'id': sequences_ids,
    'rnafm_pred': rnafm_preds,
})

atomica_results_ensemble = pd.merge(atomica_results, sequences_preds, on='id', how='left')
atomica_results_ensemble['pred_probability'] = 0.5 * atomica_results_ensemble['rnafm_pred'] + 0.5 * atomica_results_ensemble['pred']


auroc = roc_auc_score(atomica_results_ensemble['label'], atomica_results_ensemble['pred_probability'])

precision, recall, thresholds = precision_recall_curve(atomica_results_ensemble['label'], atomica_results_ensemble['pred_probability'])
auprc = auc(recall, precision)

f1 = f1_score(atomica_results_ensemble['label'], atomica_results_ensemble['pred_probability'] >= 0.3)

print("AUROC: ", auroc)
print("AUPRC: ", auprc)
print("F1: ", f1)
print("Mean label: ", np.mean(atomica_labels))

AUROC:  0.680966717090814
AUPRC:  0.2165669318746258
F1:  0.1619047619047619
Mean label:  0.07824074074074074


### Grouped scores by id

In [62]:
atomica_results['pred_bin'] = atomica_preds >= best_t

atomica_results['correct'] = atomica_results['label'] == atomica_results['pred_bin']
balanced_acc = atomica_results.groupby('id')['correct'].mean().mean()

atomica_results['baseline_correct'] = atomica_results['label'] == 0
baseline_acc = atomica_results.groupby('id')['baseline_correct'].mean().mean()

print(balanced_acc, baseline_acc)

0.8900087226463383 0.8965912059076241


In [67]:
grouped = atomica_results.groupby('id')

auprcs, aurocs = [], []
baseline = []

for _, g in grouped:
    y_true = g['label']
    y_pred = g['pred']
    # skip if only one class is present (AUROC undefined)
    if len(set(y_true)) > 1:
        precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
        auprc = auc(recall, precision)
        auroc = roc_auc_score(y_true, y_pred)
        auprcs.append(auprc)
        aurocs.append(auroc)
        baseline.append(np.mean(y_true))


mean_auprc = sum(auprcs) / len(auprcs)
mean_auroc = sum(aurocs) / len(aurocs)
mean_baseline = sum(baseline) / len(baseline)

print(f"Mean AUPRC: {mean_auprc:.4f}")
print(f"Mean AUROC: {mean_auroc:.4f}")
print(f"Baseline: {mean_baseline:.4f}")

Mean AUPRC: 0.2359
Mean AUROC: 0.6016
Baseline: 0.1034
